
# Analysing data collected from the `Data Safety` section on Google Play Store

This notebook summarizes Google Play `Data Safety` disclosures for mHealth apps using **unique app counts**.

In [ ]:

import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm

plt.style.use("default")
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 200)



## Load data and create a unique app-level table

In [ ]:
DATA_PATH = Path("../data/mhealth_apps_metrics.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError("Could not find mhealth_apps_metrics.csv in ../data")

FIG_DIR = Path("../figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

mhealth_apps = pd.read_csv(DATA_PATH)
mhealth_apps["app_id"] = mhealth_apps["app_id"].astype(str).str.strip()

TEXT_COLUMNS = [
    "data_safety_data_shared",
    "data_safety_data_collected",
    "data_safety_security_practices",
    "data_safety_shared_purpose",
    "data_safety_collected_purpose",
    "categories",
    "category",
    "app_name",
]

NUMERIC_COLUMNS = [
    "downloads",
    "downloads_int",
    "ratings_count",
    "average_score",
    "num_permissions",
    "num_dangerous_permissions",
    "num_trackers",
]

for col in NUMERIC_COLUMNS:
    if col in mhealth_apps.columns:
        mhealth_apps[col] = pd.to_numeric(mhealth_apps[col], errors="coerce")


def first_non_null(series):
    series = series.dropna()
    return series.iloc[0] if not series.empty else np.nan


def combine_unique_text(series):
    values = []
    seen = set()
    for value in series.dropna():
        value = str(value).strip()
        if not value or value.lower() == "nan":
            continue
        if value not in seen:
            seen.add(value)
            values.append(value)
    return ", ".join(values) if values else np.nan


agg_map = {col: combine_unique_text for col in TEXT_COLUMNS if col in mhealth_apps.columns}
for col in NUMERIC_COLUMNS:
    if col in mhealth_apps.columns:
        agg_map[col] = "max"

for col in ["developer_name", "content_rating", "updated_on", "free", "offersIAP"]:
    if col in mhealth_apps.columns:
        agg_map[col] = first_non_null

mhealth_apps_unique = (
    mhealth_apps
    .groupby("app_id", as_index=False)
    .agg(agg_map)
)

TOTAL_UNIQUE_APPS = mhealth_apps_unique["app_id"].nunique()

print(f"Original rows: {len(mhealth_apps):,}")
print(f"Unique app_ids in raw data: {mhealth_apps['app_id'].nunique():,}")
print(f"Rows in app-level table: {len(mhealth_apps_unique):,}")
print(f"Total unique apps used for percentages: {TOTAL_UNIQUE_APPS:,}")



## Helper functions

In [ ]:

sensitive_data = {
    "Email address": ["GDPR", "Google"],
    "Name": ["GDPR", "Google", "HIPAA"],
    "Device or other IDs": ["Google"],
    "Crash logs": ["Google"],
    "App interactions": ["Google"],
    "Diagnostics": ["Google"],
    "User IDs": ["GDPR", "Google"],
    "Fitness info": ["GDPR", "HIPAA", "Google"],
    "Photos": ["GDPR", "Google"],
    "Health info": ["GDPR", "HIPAA", "Google"],
    "Other user-generated content": ["Google"],
    "Other app performance data": ["Google"],
    "Phone number": ["GDPR", "Google", "HIPAA"],
    "Approximate location": ["GDPR", "Google"],
    "Purchase history": ["GDPR", "Google"],
    "In-app search history": ["Google"],
    "Other in-app messages": ["Google"],
    "Precise location": ["GDPR", "Google"],
    "Address": ["GDPR", "Google", "HIPAA"],
    "Videos": ["GDPR", "Google"],
    "Files/docs": ["Google"],
    "Emails": ["GDPR", "Google"],
    "Sexual orientation": ["GDPR"],
    "User payment info": ["GDPR", "Google"],
    "Contacts": ["GDPR", "Google"],
    "Installed apps": ["Google"],
    "Race/ethnicity": ["GDPR", "HIPAA"],
    "Voice or sound recordings": ["Google"],
    "Calendar events": ["Google"],
    "SMS or MMS": ["Google"],
    "Other financial info": ["GDPR", "Google"],
    "Web browsing history": ["Google"],
    "Political or religious beliefs": ["GDPR"],
}


def mark_sensitive_and_regulation(data_type):
    sensitive = "Yes" if data_type in sensitive_data else "No"
    regulations = ", ".join(sensitive_data.get(data_type, []))
    return sensitive, regulations


def normalize_data_label(value):
    replacements = {
        "Files and docs": "Files/docs",
        "Race and ethnicity": "Race/ethnicity",
        "Emails": "Email address",
    }
    value = str(value).strip().rstrip(",")
    for old, new in replacements.items():
        value = value.replace(old, new)
    return value


def split_unique_labels(cell_value):
    if pd.isna(cell_value):
        return []
    text = str(cell_value).strip()
    if not text:
        return []

    text = normalize_data_label(text)

    parts = re.split(r",\s*|\s+and\s+", text)
    labels = []
    seen = set()
    for part in parts:
        cleaned = part.strip().rstrip(",")
        if cleaned and cleaned.lower() != "nan" and cleaned not in seen:
            seen.add(cleaned)
            labels.append(cleaned)
    return labels


def summarize_app_level_labels(df, column_name):
    summary = {}

    for _, row in df[["app_id", column_name]].dropna(subset=[column_name]).iterrows():
        labels = split_unique_labels(row[column_name])
        for label in labels:
            if label not in summary:
                summary[label] = {"count": 0, "Sensitive": None, "Regulations": None}
            summary[label]["count"] += 1
            summary[label]["Sensitive"], summary[label]["Regulations"] = mark_sensitive_and_regulation(label)

    summary_items = sorted(summary.items(), key=lambda x: x[1]["count"], reverse=True)

    print(f"Number of unique labels in '{column_name}': {len(summary_items)}\n")
    print(f"{'Data Type':<35}{'App Count':<12}{'Sensitive':<12}{'Regulations'}")
    print("-" * 100)
    for label, info in summary_items:
        print(f"{label:<35}{info['count']:<12}{info['Sensitive']:<12}{info['Regulations']}")

    print("\nData Types marked as Sensitive (Yes):")
    print(f"{'Data Type':<35}{'App Count':<12}{'Sensitive':<12}{'Regulations'}")
    print("-" * 100)
    sensitive_count = 0
    for label, info in summary_items:
        if info["Sensitive"] == "Yes":
            sensitive_count += 1
            print(f"{label:<35}{info['count']:<12}{info['Sensitive']:<12}{info['Regulations']}")
    print(f"\nTotal number of sensitive data types: {sensitive_count}")

    return summary_items


def summarize_security_practices(df, column_name="data_safety_security_practices", total_apps=TOTAL_UNIQUE_APPS):
    summary = {}
    for _, row in df[["app_id", column_name]].dropna(subset=[column_name]).iterrows():
        practices = split_unique_labels(row[column_name])
        for practice in practices:
            summary[practice] = summary.get(practice, 0) + 1

    summary_df = (
        pd.DataFrame(
            [{"Data Type": key, "App Count": value, "Percentage": round(value / total_apps * 100, 2)}
             for key, value in summary.items()]
        )
        .sort_values(["App Count", "Data Type"], ascending=[False, True])
        .reset_index(drop=True)
    )

    print(f"Number of unique security practices: {len(summary_df)}\n")
    print(summary_df.to_string(index=False))
    return summary_df


## Data Shared

In [ ]:

apps_with_shared_data = mhealth_apps_unique.dropna(subset=["data_safety_data_shared"]).copy()
num_apps_shared = apps_with_shared_data["app_id"].nunique()
print(
    f"Number of unique apps that share data with third parties: "
    f"{num_apps_shared} ({num_apps_shared / TOTAL_UNIQUE_APPS * 100:.2f}%)"
)

data_shared_summary = summarize_app_level_labels(apps_with_shared_data, "data_safety_data_shared")


## Data Collected

In [ ]:

apps_with_collected_data = mhealth_apps_unique.dropna(subset=["data_safety_data_collected"]).copy()
num_apps_collected = apps_with_collected_data["app_id"].nunique()
print(
    f"Number of unique apps collecting data: "
    f"{num_apps_collected} ({num_apps_collected / TOTAL_UNIQUE_APPS * 100:.2f}%)"
)

data_collected_summary = summarize_app_level_labels(apps_with_collected_data, "data_safety_data_collected")


## Shared vs. collected data types

In [ ]:

data_types_1 = [item[0] for item in data_shared_summary]
counts_1 = [item[1]["count"] for item in data_shared_summary]

data_types_2 = [item[0] for item in data_collected_summary]
counts_2 = [item[1]["count"] for item in data_collected_summary]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 7))

colors_1 = cm.viridis(np.linspace(0, 1, len(data_types_1)))
ax1.barh(data_types_1, counts_1, color=colors_1, height=0.8)
ax1.set_xlabel("# of Unique Apps", fontsize=11)
ax1.set_title("(a) Data Types Shared", fontsize=11)
ax1.invert_yaxis()
ax1.grid(axis="x", linestyle="--", alpha=0.7)
ax1.tick_params(axis="y", pad=8)

colors_2 = cm.plasma(np.linspace(0, 1, len(data_types_2)))
ax2.barh(data_types_2, counts_2, color=colors_2, height=0.8)
ax2.set_xlabel("# of Unique Apps", fontsize=11)
ax2.set_title("(b) Data Types Collected", fontsize=11)
ax2.invert_yaxis()
ax2.grid(axis="x", linestyle="--", alpha=0.7)
ax2.tick_params(axis="y", pad=8)

plt.tight_layout()
plt.savefig(FIG_DIR / "shared_and_collected_data_types_unique_apps.png", dpi=300, bbox_inches="tight")
plt.show()


## Security Practices

In [ ]:

apps_with_security_practices = mhealth_apps_unique.dropna(subset=["data_safety_security_practices"]).copy()
num_apps_security = apps_with_security_practices["app_id"].nunique()
print(
    f"Number of unique apps mentioning security practices: "
    f"{num_apps_security} ({num_apps_security / TOTAL_UNIQUE_APPS * 100:.2f}%)"
)

security_practices_summary = summarize_security_practices(apps_with_security_practices)
